In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

In [ ]:
YEAR = 2023

In [ ]:
pd.set_option("display.max_columns", 300)

In [ ]:
dir_path = Path.cwd().parents[0]
data_folder = dir_path.joinpath("data/to_model")
model_df_path = data_folder.joinpath("model_df.parquet")


In [ ]:
df = pd.read_parquet(model_df_path)
df

In [ ]:
df_year = df.filter(regex=r"_2019_")
df_year = pd.concat([df[["lat", "long", "speciesID"]], df_year], axis=1)
df_year


In [ ]:
df['speciesID'].value_counts()

In [ ]:
SAMPLES_PER_CLASS = 3000

df_stratified = (
    df_year
    .groupby('speciesID', group_keys=True)
    .apply(lambda x: x.sample(n=min(len(x), SAMPLES_PER_CLASS), random_state=42))
    .reset_index()
).drop('level_1', axis=1)


In [ ]:
df_stratified['speciesID'].value_counts()

In [ ]:
x = df_stratified.drop(columns=['lat', "long", "speciesID"])
y = pd.DataFrame(df_stratified['speciesID'])

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,     # reproducible split
    shuffle=True
)

In [ ]:
y_train.value_counts()

### TRAIN

In [ ]:
# species = y_train.values.reshape(-1,1)

# encoder = OneHotEncoder(sparse_output=False, dtype=float)

# Y_onehot = encoder.fit_transform(species)

# # Now create DataFrame
# Y_df = pd.DataFrame(Y_onehot, columns=[f"Species_{int(cat)}" for cat in encoder.categories_[0]])

In [ ]:
x_train.to_parquet(data_folder.joinpath('X.parquet'), index=False)
y_train.to_parquet(data_folder.joinpath('Y.parquet'), index=False)

### TEST

In [ ]:
# species = y_test.values.reshape(-1,1)

# encoder = OneHotEncoder(sparse_output=False, dtype=float)

# Y_onehot = encoder.fit_transform(species)

# # Now create DataFrame
# Y_df = pd.DataFrame(Y_onehot, columns=[f"Species_{int(cat)}" for cat in encoder.categories_[0]])

In [ ]:
x_test.to_parquet(data_folder.joinpath('X_test.parquet'), index=False)
y_test.to_parquet(data_folder.joinpath('Y_test.parquet'), index=False)